# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Se implementaron y compararon dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg. También se investigaron experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Se utilizaron los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Se cargaron los textos originales.
2. Se identificaron el número de documentos y sus tamaños.
3. Se implementaron la tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Se seleccionó y documentó una configuración de preprocesamiento que se mantuvo en todos los experimentos.


In [ ]:
                                     
from collections import defaultdict
from pathlib import Path
from statistics import mean
import re
import sys
import tarfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import nltk
from nltk.corpus import gutenberg


try:
    gutenberg.fileids()
except LookupError:
    nltk.download("gutenberg")


                                                                       
PATRON_TOKEN = re.compile(r"[^\W_]+(?:['’][^\W_]+)?", flags=re.UNICODE)

def normalizar(texto):
    return PATRON_TOKEN.findall(texto.casefold())

documentos = {
    doc_id: tuple(normalizar(gutenberg.raw(doc_id)))
    for doc_id in gutenberg.fileids()
}

resumen_corpus = pd.DataFrame([
    {
        "documento": doc_id,
        "caracteres": len(gutenberg.raw(doc_id)),
        "tokens": len(tokens),
        "tokens únicos": len(set(tokens)),
    }
    for doc_id, tokens in documentos.items()
]).sort_values("documento").reset_index(drop=True)

display(Markdown(
    "### Configuración seleccionada\n\n"
    "Se convierte el texto con `casefold`, se conservan secuencias alfanuméricas "
    "y palabras con apóstrofo interno, y se descarta la puntuación. Esta misma "
    "normalización se utiliza en todos los experimentos."
))
display(resumen_corpus)
print(f"Documentos: {len(documentos):,}")
print(f"Tokens totales: {resumen_corpus['tokens'].sum():,}")


**2. Búsqueda secuencial**

Se implementó una función que recibió un término de consulta y devolvió los identificadores de todos los documentos que lo contenían mediante una búsqueda secuencial.

La implementación utilizó obligatoriamente dos lazos `for` anidados: el primero recorrió los documentos y el segundo recorrió los tokens de cada documento.

No se utilizaron índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenaran anticipadamente los documentos asociados a cada término.

Se ejecutaron consultas con términos frecuentes, poco frecuentes e inexistentes.


In [ ]:
def busqueda_secuencial(coleccion, termino):
    termino = normalizar(termino)
    if len(termino) != 1:
        raise ValueError("La consulta debe producir exactamente un término normalizado.")
    termino = termino[0]

    encontrados = []
    for doc_id, tokens in coleccion.items():                                  
        for token in tokens:                                               
            if token == termino:
                encontrados.append(doc_id)
                break
    return encontrados

consultas = ["the", "whale", "terminoinexistente123"]
resultados_secuenciales = {
    termino: busqueda_secuencial(documentos, termino)
    for termino in consultas
}

for termino, encontrados in resultados_secuenciales.items():
    print(f"{termino!r}: {len(encontrados)} documento(s) -> {encontrados}")


**3. Medición de tiempos de ejecución**

Se utilizó `time.perf_counter_ns()` para obtener el tiempo de ejecución de las consultas.

Cada medición correspondió a **30 ejecuciones de la misma consulta** y se calculó la media aritmética.

Los resultados se registraron en milisegundos. La carga de archivos y la tokenización no se incluyeron dentro del tiempo de consulta.


In [ ]:
def medir_consulta(funcion, repeticiones=30):
    tiempos_ns = []
    ultimo_resultado = None
    for _ in range(repeticiones):
        inicio = time.perf_counter_ns()
        ultimo_resultado = funcion()
        tiempos_ns.append(time.perf_counter_ns() - inicio)
    return mean(tiempos_ns) / 1_000_000, ultimo_resultado

filas_tiempo_secuencial = []
for termino in consultas:
    tiempo_ms, encontrados = medir_consulta(
        lambda t=termino: busqueda_secuencial(documentos, t)
    )
    filas_tiempo_secuencial.append({
        "consulta": termino,
        "documentos recuperados": len(encontrados),
        "tiempo medio (ms)": tiempo_ms,
        "repeticiones": 30,
    })

tiempos_secuencial = pd.DataFrame(filas_tiempo_secuencial)
display(tiempos_secuencial.round({"tiempo medio (ms)": 6}))


**4. Construcción del índice invertido**

Se implementó un índice invertido que incluyó el vocabulario de la colección y las listas de documentos asociadas con cada término.

Se inspeccionaron las listas de postings y se calculó la frecuencia documental (DF) de varios términos.

Se implementaron consultas de un término y consultas booleanas con AND, OR y NOT.


In [ ]:
def construir_indice_invertido(coleccion):
    indice = defaultdict(set)
    for doc_id, tokens in coleccion.items():
        for token in set(tokens):
            indice[token].add(doc_id)
    return dict(indice)

indice = construir_indice_invertido(documentos)
universo = set(documentos)

def busqueda_indice(indice, termino):
    tokens = normalizar(termino)
    if len(tokens) != 1:
        raise ValueError("La consulta debe producir exactamente un término normalizado.")
    return sorted(indice.get(tokens[0], set()))

def consulta_and(indice, termino_a, termino_b):
    return sorted(set(busqueda_indice(indice, termino_a)) &
                  set(busqueda_indice(indice, termino_b)))

def consulta_or(indice, termino_a, termino_b):
    return sorted(set(busqueda_indice(indice, termino_a)) |
                  set(busqueda_indice(indice, termino_b)))

def consulta_not(indice, termino, universo):
    return sorted(set(universo) - set(busqueda_indice(indice, termino)))

terminos_inspeccion = ["the", "whale", "love", "terminoinexistente123"]
postings = pd.DataFrame([
    {
        "término": termino,
        "DF": len(busqueda_indice(indice, termino)),
        "postings": busqueda_indice(indice, termino),
    }
    for termino in terminos_inspeccion
])

display(Markdown(f"El vocabulario contiene **{len(indice):,} términos**."))
display(postings)
print("AND (love AND the):", consulta_and(indice, "love", "the"))
print("OR  (whale OR love):", consulta_or(indice, "whale", "love"))
print("NOT (NOT whale):", consulta_not(indice, "whale", universo))


**5. Comparación**

Se ejecutaron las mismas consultas con ambos métodos.

Se comprobó que recuperaron exactamente los mismos identificadores de documentos y se midió el tiempo medio de 30 ejecuciones para cada consulta y método.


In [ ]:
comparacion = []
for termino in consultas:
    t_secuencial, r_secuencial = medir_consulta(
        lambda t=termino: busqueda_secuencial(documentos, t)
    )
    t_indice, r_indice = medir_consulta(
        lambda t=termino: busqueda_indice(indice, t)
    )
    comparacion.append({
        "consulta": termino,
        "resultados iguales": r_secuencial == r_indice,
        "documentos": len(r_secuencial),
        "secuencial (ms)": t_secuencial,
        "índice (ms)": t_indice,
        "aceleración": t_secuencial / t_indice if t_indice else np.inf,
    })

comparacion_pequena = pd.DataFrame(comparacion)
assert comparacion_pequena["resultados iguales"].all()
display(comparacion_pequena.round({
    "secuencial (ms)": 6, "índice (ms)": 6, "aceleración": 2,
}))
display(Markdown(
    "Los dos métodos recuperan exactamente los mismos documentos. La diferencia "
    "es que la búsqueda secuencial vuelve a recorrer el corpus, mientras el índice "
    "consulta una estructura construida previamente."
))


### Parte B. Corpus grande

**6. Corpus de Gutenberg**

Se utilizó el archivo oficial `txt-files.tar`, obtenido a partir de `txt-files.tar.zip`, disponible en:

https://www.gutenberg.org/cache/epub/feeds/

Se identificaron los archivos TXT, se evitó contar distintas versiones del mismo libro como documentos independientes y se registró la cantidad de documentos válidos.


In [ ]:
                                                               
BASE = Path.cwd()
candidatos = [
    BASE / "txt-files.tar",
    Path("txt-files.tar"),
    Path(r"D:\EPN\Septimo\Recuperacion de Informacion\Cuadernos\txt-files.tar"),
]
RUTA_TAR = next((ruta.resolve() for ruta in candidatos if ruta.exists()), None)
if RUTA_TAR is None:
    raise FileNotFoundError("No se encontró txt-files.tar en la carpeta del cuaderno.")


                                                                          
patron_gutenberg = re.compile(
    r"^cache/epub/(\d+)/pg\1(?:-([a-z0-9]+))?\.txt$", re.IGNORECASE
)

def prioridad_version(nombre, variante):
    if variante is None:
        return (0, len(nombre), nombre)
    if variante == "0":
        return (1, len(nombre), nombre)
    return (2, len(nombre), nombre)

archivos_por_id = {}
miembros_txt = 0
inicio = time.perf_counter()
with tarfile.open(RUTA_TAR, mode="r:") as tar:
    for miembro in tar:
        if not miembro.isfile():
            continue
        coincidencia = patron_gutenberg.match(miembro.name)
        if coincidencia is None:
            continue
        miembros_txt += 1
        doc_id = int(coincidencia.group(1))
        variante = coincidencia.group(2)
        candidato = (
            prioridad_version(miembro.name, variante),
            miembro.name,
            miembro.size,
            miembro.offset_data,
        )
        if doc_id not in archivos_por_id or candidato[0] < archivos_por_id[doc_id][0]:
            archivos_por_id[doc_id] = candidato

tiempo_inventario = time.perf_counter() - inicio
catalogo = pd.DataFrame([
    {
        "documento": doc_id,
        "miembro": datos[1],
        "bytes": datos[2],
        "desplazamiento": datos[3],
    }
    for doc_id, datos in sorted(archivos_por_id.items())
])

display(catalogo.head())
print(f"Archivos TXT que cumplen el patrón: {miembros_txt:,}")
print(f"Libros únicos válidos: {len(catalogo):,}")
print(f"Versiones descartadas: {miembros_txt - len(catalogo):,}")
print(f"Tiempo de inventario: {tiempo_inventario:.2f} s")


**7. Recuperación sobre el corpus grande**

Se reutilizaron las funciones desarrolladas en la Parte A.

1. Se procesaron los documentos con la misma configuración de normalización.
2. Se ejecutaron las consultas mediante búsqueda secuencial y se midieron los tiempos.
3. Se construyó el índice invertido.
4. Se repitieron las consultas mediante el índice.
5. Se comprobó la igualdad de resultados y se compararon los tiempos.

Se trabajó con subconjuntos de la colección de tamaños crecientes y se conservaron los identificadores originales. También se documentaron los recursos de memoria y almacenamiento utilizados.


In [ ]:
                                                             
MAX_DOCUMENTOS = min(1000, len(catalogo))
documentos_grandes = {}
inicio = time.perf_counter()
seleccion = catalogo.head(MAX_DOCUMENTOS).sort_values("desplazamiento")
with RUTA_TAR.open("rb") as archivo_tar:
    for fila in seleccion.itertuples():
        archivo_tar.seek(fila.desplazamiento)
        texto = archivo_tar.read(fila.bytes).decode("utf-8", errors="replace")

        documentos_grandes[int(fila.documento)] = tuple(
            sys.intern(t) for t in normalizar(texto)
        )

documentos_grandes = dict(sorted(documentos_grandes.items()))
tiempo_carga_grande = time.perf_counter() - inicio
tokens_grandes = sum(map(len, documentos_grandes.values()))

inicio = time.perf_counter()
indice_grande = construir_indice_invertido(documentos_grandes)
tiempo_indice_grande = time.perf_counter() - inicio

consultas_grandes = ["the", "love"]
filas_grande = []
for termino in consultas_grandes:
    ts, rs = medir_consulta(lambda t=termino: busqueda_secuencial(documentos_grandes, t))
    ti, ri = medir_consulta(lambda t=termino: busqueda_indice(indice_grande, t))
    filas_grande.append({
        "consulta": termino,
        "documentos": len(rs),
        "resultados iguales": rs == ri,
        "secuencial (ms)": ts,
        "índice (ms)": ti,
    })

comparacion_grande = pd.DataFrame(filas_grande)
assert comparacion_grande["resultados iguales"].all()

recursos = pd.DataFrame([{
    "documentos procesados": len(documentos_grandes),
    "tokens": tokens_grandes,
    "tamaño TAR (GB)": RUTA_TAR.stat().st_size / 1024**3,
    "tiempo de lectura y tokenización (s)": tiempo_carga_grande,
    "tiempo de construcción del índice (s)": tiempo_indice_grande,
    "vocabulario": len(indice_grande),
}])
display(recursos.round({
    "tamaño TAR (GB)": 2,
    "tiempo de lectura y tokenización (s)": 2,
    "tiempo de construcción del índice (s)": 2,
}))
display(comparacion_grande.round({
    "secuencial (ms)": 6, "índice (ms)": 6,
}))


### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Se seleccionaron cinco tamaños crecientes del corpus grande: 50, 100, 250, 500 y 1.000 documentos.

Para cada tamaño $n$:

1. Se ejecutó una misma consulta mediante búsqueda secuencial.
2. Se repitió 30 veces y se calculó el tiempo promedio.
3. Se construyó el índice invertido correspondiente a ese subconjunto.
4. Se ejecutó la consulta mediante el índice, también 30 veces.
5. Se registraron ambos tiempos y el número total de tokens.

Se elaboró una **gráfica del tiempo medio de ejecución en función del número de documentos**, con el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Los dos métodos se representaron en la misma gráfica.


In [ ]:
tamanos_solicitados = [50, 100, 250, 500, 1000]
tamanos = [n for n in tamanos_solicitados if n <= len(documentos_grandes)]
if len(tamanos) < 5:
    raise ValueError("Se necesitan al menos 1.000 documentos para evaluar cinco tamaños.")

TERMINO_ESCALABILIDAD = "the"
filas_escalabilidad = []
for n in tamanos:
    subconjunto = dict(list(documentos_grandes.items())[:n])
    total_tokens = sum(map(len, subconjunto.values()))

    inicio = time.perf_counter()
    indice_n = construir_indice_invertido(subconjunto)
    tiempo_construccion_ms = (time.perf_counter() - inicio) * 1000

    ts, rs = medir_consulta(
        lambda c=subconjunto: busqueda_secuencial(c, TERMINO_ESCALABILIDAD)
    )
    ti, ri = medir_consulta(
        lambda idx=indice_n: busqueda_indice(idx, TERMINO_ESCALABILIDAD)
    )
    assert rs == ri
    filas_escalabilidad.append({
        "documentos": n,
        "tokens": total_tokens,
        "secuencial (ms)": ts,
        "índice (ms)": ti,
        "construcción índice (ms)": tiempo_construccion_ms,
        "vocabulario": len(indice_n),
    })

escalabilidad = pd.DataFrame(filas_escalabilidad)
display(escalabilidad.round({
    "secuencial (ms)": 6,
    "índice (ms)": 6,
    "construcción índice (ms)": 2,
}))

ax = escalabilidad.plot(
    x="documentos", y=["secuencial (ms)", "índice (ms)"],
    marker="o", figsize=(9, 5)
)
ax.set_title(f"Tiempo medio de consulta para {TERMINO_ESCALABILIDAD!r} (30 repeticiones)")
ax.set_xlabel("Número de documentos")
ax.set_ylabel("Tiempo medio (ms)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**9. Estimación de una función de tiempo**

Con los resultados experimentales se determinó una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, se utilizó un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Se estimaron los parámetros $a$ y $b$, se reportó el coeficiente de determinación $R^2$ y se representaron las funciones ajustadas junto con los datos experimentales.

Finalmente, se explicó si el modelo lineal representó adecuadamente las mediciones y cómo se relacionó con la complejidad computacional de cada algoritmo.


In [ ]:
def ajuste_lineal(x, y):
    a, b = np.polyfit(x, y, deg=1)
    estimado = a * x + b
    ss_res = np.sum((y - estimado) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot else 1.0
    return a, b, r2, estimado

x = escalabilidad["documentos"].to_numpy(dtype=float)
ajustes = []

fig, ax = plt.subplots(figsize=(9, 5))
colores = {"secuencial (ms)": "tab:blue", "índice (ms)": "tab:orange"}
for metodo in ["secuencial (ms)", "índice (ms)"]:
    y = escalabilidad[metodo].to_numpy(dtype=float)
    a, b, r2, estimado = ajuste_lineal(x, y)
    ajustes.append({"método": metodo, "a": a, "b": b, "R²": r2})
    ax.scatter(x, y, color=colores[metodo], label=f"{metodo} - datos")
    ax.plot(x, estimado, "--", color=colores[metodo],
            label=f"{metodo}: t(n)={a:.6g}n+{b:.6g}; R²={r2:.4f}")

ax.set_title("Datos experimentales y ajustes lineales")
ax.set_xlabel("Número de documentos (n)")
ax.set_ylabel("Tiempo medio (ms)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

ajustes_df = pd.DataFrame(ajustes)
display(ajustes_df.round({"a": 8, "b": 8, "R²": 5}))

seq = ajustes_df.loc[ajustes_df["método"] == "secuencial (ms)"].iloc[0]
idx = ajustes_df.loc[ajustes_df["método"] == "índice (ms)"].iloc[0]

def calidad(r2):
    if r2 >= 0.90:
        return "representa adecuadamente la tendencia observada"
    if r2 >= 0.70:
        return "representa de forma aproximada la tendencia, aunque existe variabilidad"
    return "no representa bien las mediciones; el ruido y los costos constantes tienen bastante peso"

display(Markdown(f"""### Interpretación

Para la búsqueda secuencial se obtuvo
$t(n)={seq['a']:.8f}n+{seq['b']:.8f}$ ms, con $R^2={seq['R²']:.5f}$; por tanto, el modelo {calidad(seq['R²'])}.

Para la consulta mediante índice se obtuvo
$t(n)={idx['a']:.8f}n+{idx['b']:.8f}$ ms, con $R^2={idx['R²']:.5f}$; por tanto, el modelo {calidad(idx['R²'])}.

La búsqueda secuencial tiene complejidad $O(T)$, donde $T$ es la cantidad de tokens que se deben revisar, por lo que su tiempo tiende a crecer con el corpus. La consulta de un término en el índice se aproxima a $O(1 + df)$: el acceso al diccionario es constante en promedio y el costo restante depende del tamaño de la lista de postings. El ajuste lineal del índice puede tener un $R^2$ bajo porque sus tiempos son muy pequeños y el ruido de medición domina. Esta ventaja de consulta no es gratuita: antes fue necesario construir el índice, cuyo tiempo y memoria crecen con la colección.
"""))
